In [94]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
import os 
from pydantic import BaseModel,Field
from langchain_core.output_parsers import PydanticOutputParser
import operator
from langchain_core.prompts import PromptTemplate




In [96]:
## getting ready an llM which will supprt the structured outputr for all the task :
llm=HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="conversational"
)

model=ChatHuggingFace(llm=llm)

In [ ]:
## variying model working or not 


In [97]:
## creating pydantic schema for structured output from that above llm :

class ModelSchema(BaseModel):
    feedback:str=Field(description="generate feedback for the provided eassy")
    score:int=Field(description="generate an score between 0 to 10 only")
    


In [100]:
## getting structured model

parser=PydanticOutputParser(pydantic_object=ModelSchema)



In [99]:
## defining state:

class EassySchema(TypedDict):
    eassy_text:str
    cot_feedback:str
    doa_feedback:str
    language_feedback:str
    individual_score:Annotated[list[int],operator.add]
    summarized_feedback:str
    average_score:float



In [101]:
## node definations 
def clarity_of_thought(state:EassySchema):
    prompt=PromptTemplate(
        template=""" 
    You are an expert essay evaluator.

Your task is to evaluate the given essay strictly based on "clarity of thought".

Clarity of thought includes:
- Logical flow of ideas
- Coherence between sentences and paragraphs
- Clear expression of arguments
- Absence of ambiguity or confusion
- Proper structuring (introduction, body, conclusion)

Evaluation Guidelines:
- Focus ONLY on clarity of thought (ignore grammar, spelling, or creativity unless they affect clarity)
- Be objective and precise
- Do not be overly harsh or overly lenient

Scoring:
- Give a score between 0 to 10
- 0–3 → Poor clarity (confusing, disorganized)
- 4–6 → Average clarity (some structure but unclear in parts)
- 7–8 → Good clarity (mostly clear and logical)
- 9–10 → Excellent clarity (very clear, well-structured, easy to follow)

Output Format (STRICT):
Return ONLY valid JSON. Do not include any explanation outside JSON.

{format_instructions}

Now evaluate the following essay:

{eassy}

""",
input_variables=["eassy"],
partial_variables={
    "format_instructions":parser.get_format_instructions()
}
    )
    eassy_text=state['eassy_text']
    final_prompt=prompt.invoke({'eassy':eassy_text})
    response=model.invoke(final_prompt)
    parsed=parser.parse(response.content)

    return {'cot_feedback':parsed.feedback,'individual_score':[parsed.score]}

 
    


def depth_of_analysis(state:EassySchema):
    prompt=PromptTemplate(
        template=""" 
    You are an expert essay evaluator.

Your task is to evaluate the given essay strictly based on "depth of analysis".

Depth of analysis includes:
- Strength and depth of arguments
- Use of reasoning (not just statements, but explanations and justification)
- Ability to explore ideas beyond surface level
- Use of examples, evidence, or reasoning to support claims
- Critical thinking (analysis, not just description)

Evaluation Guidelines:
- Focus ONLY on depth of analysis (ignore grammar, spelling, or structure unless they affect reasoning)
- Do not reward length unless it adds meaningful insight
- Penalize shallow, repetitive, or purely descriptive content

Scoring:
- Give a score between 0 to 10
- 0–3 → Very shallow (no real analysis, only statements)
- 4–6 → Basic analysis (some reasoning but limited depth)
- 7–8 → Good analysis (clear reasoning with supporting ideas)
- 9–10 → Excellent analysis (deep insight, strong reasoning, well-supported arguments)

Output Format (STRICT):
Return ONLY valid JSON. Do not include any explanation outside JSON.

{format_instructions}

Now evaluate the following essay:

{essay}
""",
input_variables=["essay"],
partial_variables={
    "format_instructions":parser.get_format_instructions()
}
    )
    essay_text=state['eassy_text']
    final_prompt=prompt.invoke({'essay':essay_text})
    response=model.invoke(final_prompt)
    parsed=parser.parse(response.content)

    return {'doa_feedback':parsed.feedback,'individual_score':[parsed.score]}


def language(state:EassySchema):
    prompt=PromptTemplate(
        template=""" 
    You are an expert English language evaluator.

Your task is to evaluate the given essay strictly based on "language correctness and quality".

Language verification includes:
- Grammar accuracy (tense, subject-verb agreement, sentence formation)
- Vocabulary usage (appropriate, precise, and varied word choice)
- Sentence structure (well-formed and readable sentences)
- Spelling and punctuation
- Fluency and natural flow of language

Evaluation Guidelines:
- Focus ONLY on language quality (ignore content quality, logic, or depth unless they affect language)
- Penalize grammatical errors, awkward phrasing, and incorrect word usage
- Reward clear, fluent, and natural writing

Scoring:
- Give a score between 0 to 10
- 0–3 → Poor language (frequent grammar mistakes, hard to understand)
- 4–6 → Average language (some errors, but understandable)
- 7–8 → Good language (minor errors, mostly fluent)
- 9–10 → Excellent language (almost no errors, very natural and fluent)

Output Format (STRICT):
Return ONLY valid JSON. Do not include any explanation outside JSON.

{format_instructions}

Now evaluate the following essay:

{essay}
""",
input_variables=["essay"],
partial_variables={"format_instructions":parser.get_format_instructions()
                   }

    )
    essay_text=state['eassy_text']
    final_prompt=prompt.invoke({'essay':essay_text})
    response=model.invoke(final_prompt)
    parsed=parser.parse(response.content)
    return {'language_feedback':parsed.feedback,'individual_score':[parsed.score]}

def final_eval(state:EassySchema):
    prompt=f"""
         based on this clarity of tought feedback : {state['cot_feedback']}\n
         and based on this depth of analysis feedback: {state['doa_feedback']}\n 
         and based on this language feedback:{state['language_feedback']} \n 
        
     genrerate an summarized feedback based on above mention feedbacks 

"""
    summarized_feedback=model.invoke(prompt).content
    avg_score=sum(state['individual_score'])/len(state['individual_score'])
    state['summarized_feedback']=summarized_feedback
    state['average_score']=avg_score
    return {'summarized_feedback':summarized_feedback,'average_score':avg_score}
    


In [102]:
## creating graph ::--> 

graph=StateGraph(EassySchema)

##adding nodes to then flow

graph.add_node("COT",clarity_of_thought)
graph.add_node("DOA",depth_of_analysis)
graph.add_node("lang",language)
graph.add_node("final",final_eval)

## adding edges to the graph

graph.add_edge(START,"COT")
graph.add_edge(START,"DOA")
graph.add_edge(START,"lang")

graph.add_edge("COT","final")
graph.add_edge("DOA","final")
graph.add_edge("lang","final")

graph.add_edge("final",END)

workflow=graph.compile()




In [103]:
text=f""" 
Artificial Intelligence (AI) is rapidly transforming the world, and India is emerging as one of the most promising nations in this technological revolution. With its vast population, growing digital infrastructure, talented workforce, and expanding startup ecosystem, India is uniquely positioned to benefit from AI-driven innovation. The AI revolution in India is not only changing industries but also reshaping education, healthcare, agriculture, governance, and employment.

One of the major reasons behind the growth of AI in India is the rapid expansion of digital technology. Over the last decade, internet accessibility and smartphone usage have increased significantly, allowing millions of people to connect to digital platforms. Government initiatives such as Digital India have further accelerated technological adoption. As a result, enormous amounts of data are being generated daily, which acts as fuel for AI systems. Companies and organizations are now using this data to improve services, predict customer behavior, and automate complex tasks.

The healthcare sector in India is experiencing major improvements through AI applications. AI-powered systems are helping doctors detect diseases at early stages, analyze medical reports, and recommend treatments more accurately. In rural areas where access to doctors is limited, AI-based telemedicine platforms are providing medical assistance remotely. This can significantly reduce healthcare inequality and improve the quality of life for millions of people.

Agriculture, which employs a large portion of India’s population, is also being transformed by AI. Farmers are using AI tools to predict weather conditions, monitor crop health, and optimize irrigation. These technologies help farmers make informed decisions, reduce losses, and increase productivity. In a country where agriculture is highly dependent on unpredictable weather patterns, AI can play a critical role in ensuring food security and economic stability.

Education is another sector benefiting from the AI revolution. Personalized learning platforms powered by AI can identify the strengths and weaknesses of students and provide customized learning experiences. This is particularly important in India, where the quality of education varies significantly across regions. AI has the potential to bridge this gap by making high-quality learning resources accessible to students regardless of their location.

India’s startup ecosystem has also embraced AI enthusiastically. Many startups are developing innovative AI-based solutions in finance, cybersecurity, logistics, and e-commerce. Large technology companies are investing heavily in AI research and development centers in India due to the availability of skilled engineers and data scientists. This growth is creating new career opportunities and encouraging young people to pursue careers in AI and machine learning.

However, the AI revolution in India also brings several challenges. One major concern is job displacement caused by automation. As machines become capable of performing repetitive tasks, some traditional jobs may disappear. This makes skill development and reskilling extremely important. The education system and government policies must adapt to prepare workers for future industries. Ethical concerns such as data privacy, algorithmic bias, and misuse of AI technologies also require careful regulation and responsible implementation.

Despite these challenges, the future of AI in India appears highly promising. If India successfully combines technological advancement with ethical governance and inclusive growth, AI can become a powerful tool for national development. Rather than replacing humans entirely, AI should be viewed as a technology that enhances human capabilities and improves efficiency.

In conclusion, the AI revolution in India is creating opportunities that were unimaginable a few decades ago. From healthcare and agriculture to education and business, AI is transforming nearly every aspect of society. While challenges exist, proper planning, education, and responsible innovation can ensure that the benefits of AI reach all sections of society. India has the potential not only to become a major consumer of AI technology but also a global leader in AI innovation and research.
"""

In [104]:
initial_state={'eassy_text':text}

final_state=workflow.invoke(initial_state)

In [108]:
## printing different values of the state::

final_state

{'eassy_text': ' \nArtificial Intelligence (AI) is rapidly transforming the world, and India is emerging as one of the most promising nations in this technological revolution. With its vast population, growing digital infrastructure, talented workforce, and expanding startup ecosystem, India is uniquely positioned to benefit from AI-driven innovation. The AI revolution in India is not only changing industries but also reshaping education, healthcare, agriculture, governance, and employment.\n\nOne of the major reasons behind the growth of AI in India is the rapid expansion of digital technology. Over the last decade, internet accessibility and smartphone usage have increased significantly, allowing millions of people to connect to digital platforms. Government initiatives such as Digital India have further accelerated technological adoption. As a result, enormous amounts of data are being generated daily, which acts as fuel for AI systems. Companies and organizations are now using this

In [109]:
final_state['cot_feedback']

'The essay demonstrates excellent clarity of thought with a logical flow of ideas, coherence between sentences and paragraphs, clear expression of arguments, and proper structuring.'

In [110]:
final_state['doa_feedback']

'The essay provides a comprehensive overview of the AI revolution in India, touching on various sectors and challenges. However, the analysis lacks depth, and the arguments are often based on stated facts rather than critical thinking.'

In [111]:
final_state['language_feedback']

'Overall, the essay demonstrates good language quality, with mostly clear, fluent, and natural writing. However, a few minor errors and awkward phrasing detract from perfection.'

In [112]:
final_state['summarized_feedback']

"Here's a summarized feedback based on the given feedbacks:\n\n**Overall Feedback**\n\nYour essay demonstrates strong clarity of thought, a logical flow of ideas, and good language quality. However, there is room for improvement in providing depth in analysis and developing arguments based on critical thinking rather than just stated facts.\n\n**Key Areas for Improvement**\n\n1. **Depth of Analysis**: While you provide a comprehensive overview of the AI revolution in India, your analysis could benefit from more critical thinking and in-depth exploration of the topics.\n2. **Critical Thinking**: Your arguments are often based on stated facts, rather than analyzing and interpreting the information to present more nuanced and insightful perspectives.\n3. **Language Quality**: Although your writing is mostly clear, fluent, and natural, there are minor errors and awkward phrasing that detract from the overall quality of the essay.\n\n**Recommendations**\n\n* Develop more in-depth analysis a

In [113]:
final_state['individual_score']

[9, 5, 8]

In [114]:
final_state['average_score']

7.333333333333333